In [4]:
import os

PROJECT_ID = "second-capsule-472911-g7"
BUCKET_NAME = "mlops-course-second-capsule-472911-g7-unique"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["BUCKET_NAME"] = BUCKET_NAME

!gcloud auth list
!gsutil ls gs://$BUCKET_NAME || echo "Bucket empty or new"

                  Credentialed Accounts
ACTIVE  ACCOUNT
*       968381759525-compute@developer.gserviceaccount.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

gs://mlops-course-second-capsule-472911-g7-unique/registry.db
gs://mlops-course-second-capsule-472911-g7-unique/data/
gs://mlops-course-second-capsule-472911-g7-unique/dvcstore/
gs://mlops-course-second-capsule-472911-g7-unique/models/


In [5]:
!git config --global user.email "rajyalakshmijampani@gmail.com"
!git config --global user.name "Rajya Lakshmi"

In [6]:
os.chdir("/home/jupyter/mlops")

In [7]:
!git checkout v1-version

M	k8s/deployment.yaml
Already on 'v1-version'


In [8]:
!dvc pull -r mygcs
!dvc checkout

Fetching
!
  0% Checking cache in '/home/jupyter/mlops/.dvc/cache/files/md5'| |0/? [00:00<?
Fetching                                                                        
Building workspace index                              |4.00 [00:00,  691entry/s]
Comparing indexes                                    |5.00 [00:00, 2.00kentry/s]
Applying changes                                      |0.00 [00:00,     ?file/s]
Everything is up to date.
Building workspace index                              |4.00 [00:00, 36.6entry/s]
Comparing indexes                                    |5.00 [00:00, 2.25kentry/s]
Applying changes                                      |0.00 [00:00,     ?file/s]


In [9]:
!ls model.joblib data.csv # check files available or not

data.csv  model.joblib


In [10]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
joblib
numpy
scikit-learn
pandas

Overwriting requirements.txt


In [11]:
%%writefile Dockerfile
# 1. Use official Python base image
FROM python:3.10-slim

# 2. Set working directory
WORKDIR /app

# 3. Copy files
COPY app/ ./app/
COPY model.joblib ./model.joblib
COPY requirements.txt .

# 4. Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# 5. Expose port
EXPOSE 8080

# 6. Command to run the server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]


Overwriting Dockerfile


In [12]:
os.makedirs("app", exist_ok=True)
os.makedirs("k8s", exist_ok=True)

In [34]:
%%writefile app/main.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import pandas as pd

app = FastAPI()

model = joblib.load("model.joblib")

class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def home():
    return {"message": "Welcome to the IRIS classifier."}

@app.post("/predict")
def predict(data: IrisInput):
    x = pd.DataFrame([data.dict()])
    pred = model.predict(x)
    return {"Predicted species": pred[0]}


Overwriting app/main.py


In [25]:
%%writefile k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api
spec:
  replicas: 1
  selector:
    matchLabels:
      app: iris-api
  template:
    metadata:
      labels:
        app: iris-api
    spec:
      containers:
        - name: iris-api
          image: us-central1-docker.pkg.dev/second-capsule-472911-g7/iris-repo/iris-api:latest
          imagePullPolicy: Always
          ports:
            - containerPort: 8080
---
apiVersion: v1
kind: Service
metadata:
  name: iris-api-svc
spec:
  type: LoadBalancer
  selector:
    app: iris-api
  ports:
    - port: 80
      targetPort: 8080

Overwriting k8s/deployment.yaml


Enable APIs:

GCP -> APIs and Services -> Library 

Artifact Registry API - To store Docker images
Kubernetes Engine API - To create & manage GKE clusters
Cloud Resource Manager API - For GitHub Actions to access project info


Create Artifact Registry:
Artifact Registry → Create Repository iris-repo

Create GKE cluster
Kubernetes Engine → Clusters → Create -> Standard cluster

Service Account already existing -> Added roles Artifact Registry Writer, Kubernetes Engine Developer

ci.yml should now run tests → builds Docker → pushes to Artifact Registry → deploys to GKE.

In [31]:
%%writefile .github/workflows/ci.yml
name: CI-CD

permissions:
  contents: write
  pull-requests: write
  id-token: write

on:
  push:
  pull_request:

env:
  PROJECT_ID: ${{ secrets.GCP_PROJECT }}
  ARTIFACT_REGION: ${{ secrets.ARTIFACT_REGION }}
  ARTIFACT_REPO: ${{ secrets.ARTIFACT_REPO }}
  GKE_CLUSTER: ${{ secrets.GKE_CLUSTER }}
  GKE_ZONE: ${{ secrets.GKE_ZONE }}

jobs:
  # ---------------------
  # Test Job
  # ---------------------
  test:
    runs-on: ubuntu-latest
    env:
      MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}

    steps:
      - uses: actions/checkout@v3

      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'

      - uses: iterative/setup-dvc@v1
        with:
          version: 3.63.0

      - name: Install dependencies
        run: |
          pip install pytest pandas joblib scikit-learn mlflow

      - name: Authenticate GCP
        env:
          GCP_SA_KEY: ${{ secrets.GCP_SA_KEY }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
        run: |
          echo "$GCP_SA_KEY" > /tmp/key.json
          gcloud auth activate-service-account --key-file=/tmp/key.json
          gcloud config set project $PROJECT_ID
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json

      - name: Pull data via DVC
        run: |
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json
          dvc pull -r mygcs --force

      - name: Run tests
        run: pytest -v -s tests/test_pipeline_mlflow.py --uri "$MLFLOW_TRACKING_URI" > report.txt

      - name: Set up CML
        uses: iterative/setup-cml@v2

      - name: Post CML comment
        env:
          REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        run: |
          echo "### Pytest Report" > report.md
          cat report.txt >> report.md
          cml comment create report.md


  # ---------------------
  # Build Docker image and push to Artifact Registry
  # ---------------------
  build:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v3
    
      - uses: iterative/setup-dvc@v1
        with:
          version: 3.63.0

      - name: Authenticate GCP
        env:
          GCP_SA_KEY: ${{ secrets.GCP_SA_KEY }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
        run: |
          echo "$GCP_SA_KEY" > /tmp/key.json
          gcloud auth activate-service-account --key-file=/tmp/key.json
          gcloud config set project $PROJECT_ID
          gcloud auth configure-docker ${{ secrets.ARTIFACT_REGION }}-docker.pkg.dev --quiet
        
      - name: Pull data via DVC
        run: |
          export GOOGLE_APPLICATION_CREDENTIALS=/tmp/key.json
          dvc pull -r mygcs --force

      - name: Build and Push Docker image
        env:
          ARTIFACT_REGION: ${{ secrets.ARTIFACT_REGION }}
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
          ARTIFACT_REPO: ${{ secrets.ARTIFACT_REPO }}
          IMAGE_NAME: iris-api
        run: |
          IMAGE_URI=${ARTIFACT_REGION}-docker.pkg.dev/${PROJECT_ID}/${ARTIFACT_REPO}/${IMAGE_NAME}:latest
          docker build -t $IMAGE_URI .
          docker push $IMAGE_URI
          echo "IMAGE_URI=$IMAGE_URI" >> $GITHUB_ENV


  # ---------------------
  # Deploy to GKE
  # ---------------------
  deploy:
    runs-on: ubuntu-latest
    needs: build
    steps:
      - uses: actions/checkout@v3

      - name: Authenticate to Google Cloud
        uses: google-github-actions/auth@v2
        with:
         credentials_json: ${{ secrets.GCP_SA_KEY }}
        
      - name: Connect to GKE cluster
        uses: google-github-actions/get-gke-credentials@v2
        with:
         cluster_name: ${{ secrets.GKE_CLUSTER }}
         location: ${{ secrets.GKE_ZONE }}
         project_id: ${{ secrets.PROJECT_ID }}
          
      - name: Deploy new image to GKE
        env:
          IMAGE_URI: ${{ secrets.ARTIFACT_REGION }}-docker.pkg.dev/${{ secrets.PROJECT_ID }}/${{ secrets.ARTIFACT_REPO }}/iris-api:latest
          PROJECT_ID: ${{ secrets.PROJECT_ID }}
        run: |
          # Apply manifest if not already present
          kubectl apply -f k8s/deployment.yaml --validate=false
          # Update deployment image
          kubectl set image deployment/iris-api iris-api=$IMAGE_URI
          kubectl rollout status deployment/iris-api

Overwriting .github/workflows/ci.yml


In [35]:
!git add .
!git commit -m "CD pipeline update"

[v1-version 5748606] CD pipeline update
 1 file changed, 1 insertion(+)


In [36]:
!git push origin v1-version

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 368 bytes | 368.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/rajyalakshmijampani/mlops.git
   5184e0c..5748606  v1-version -> v1-version


In [3]:
## Test fast api

# uvicorn app.main:app --reload --host 0.0.0.0